# Tome 3, Chapter 8: Markov Chains (Python companion)

Exercises first, solutions at the end. We compute stationary laws three ways, rank the pages of a synthetic web with PageRank, sample the Ising model of a magnet with the Metropolis algorithm and watch it magnetise below the critical temperature, measure mixing times on a cycle, and test card shuffles: random transpositions by their fixed points, riffle shuffles exactly with the Bayer–Diaconis formula.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(8)

## Exercises

**P8.1 (★) Three ways to a stationary law.** For the chain of Figure 8.1, compute $\pi$ (a) as a left eigenvector of $P$, (b) as the limit of $\lambda P^n$, and (c) as the fraction of time spent in each state along one long simulated path (Theorem 8.13). Check Kac's formula $\mathbb E_1[T_1]=\frac{11}4$ by simulating return times.

**P8.2 (★) The knight.** Simulate a random knight from a corner and measure its mean return time (Exercise 8.2). Also measure the fraction of time it spends in the corner during a long walk, and compare with $\frac2{336}$.

**P8.3 (★★) PageRank.** Build a random directed ``web'' of $2000$ pages by preferential attachment (each new page links to $3$ earlier pages chosen with probability proportional to their in-degree plus one). Compute PageRank with damping $\alpha=0.85$ by power iteration (pages without outgoing links jump uniformly), check that the error decreases like $\alpha^t$ (Exercise 8.4), and compare the PageRank ranking with the ranking by number of incoming links.

**P8.4 (★★) The Ising model by Metropolis.** On an $L\times L$ torus put spins $s_i=\pm1$ with $\pi(s)\propto e^{\sum_{i\sim j}s_is_j/T}$. Run the Metropolis algorithm (Exercise 8.5), flipping one spin at a time, for $L=32$ and temperatures $T$ from $1.5$ to $3.5$, and plot the mean absolute magnetisation $|\frac1{L^2}\sum s_i|$. Compare with Onsager's exact value $(1-\sinh^{-4}(2/T))^{1/8}$ below $T_c=\frac2{\ln(1+\sqrt2)}\approx2.269$.

**P8.5 (★) Mixing on a cycle.** Compute $d(t)$ exactly for the lazy walk on $\Z/n\Z$ for $n=8,16,\dots,256$, find $t_{\mathrm{mix}}$, and check the bounds of Theorem 8.17. What is the limit of $\frac{t_{\mathrm{mix}}}{n^2}$?

**P8.6 (★★) Random transpositions.** For $n=52$, simulate the shuffle and record the number of fixed points after $t$ steps. The total variation distance between the law of this statistic and its law under a uniform permutation is a lower bound for $d(t)$. Plot this lower bound against $t$, and locate it with respect to $\frac12n\ln n\approx103$ and Broder's bound $2nH_n\approx471$ (Theorem 8.20).

**P8.7 (★★) Riffle shuffles.** Compute the exact distance $d_k$ after $k$ riffle shuffles for decks of $n=13,26,52,104,208$ cards, and compare the number of shuffles needed to get below $\frac14$ with $\frac32\log_2n$. Then simulate GSR shuffles of $52$ cards and compare the empirical law of the number of rising sequences after $k=4$ shuffles with Theorem 8.23.

## Solutions

**P8.1.**

In [ ]:
P = np.array([[1/2, 1/2, 0], [1/4, 1/4, 1/2], [1/3, 1/3, 1/3]])
w, V = np.linalg.eig(P.T)
v = np.real(V[:, np.argmin(abs(w - 1))]); v /= v.sum()
print('(a) eigenvector     ', v, '  exact', [4/11, 4/11, 3/11])
print('(b) lambda P^50     ', np.array([1, 0, 0]) @ np.linalg.matrix_power(P, 50))
cum = P.cumsum(axis=1)
def path(x0, n):
    xs = np.empty(n, dtype=int); x = x0
    U = rng.random(n)
    for k in range(n):
        xs[k] = x
        x = int(np.searchsorted(cum[x], U[k], side='right'))
    return xs
xs = path(0, 200000)
print('(c) occupation      ', np.bincount(xs, minlength=3) / len(xs))
visits = np.flatnonzero(xs == 0)
print('mean return time to state 1:', np.diff(visits).mean(), '  Kac: 11/4 =', 11/4)
print('eigenvalues of P:', np.round(np.sort(w.real), 4))

All three methods agree with $(\frac4{11},\frac4{11},\frac3{11})$. The other eigenvalues are $0$ and $\frac1{12}$, which is why $\lambda P^{50}$ is exact to machine precision: here the convergence is very fast. Note that this chain is not reversible ($\pi(1)P(1,3)=0$ but $\pi(3)P(3,1)>0$), and its eigenvalues need not be real; they happen to be here.

**P8.2.**

In [ ]:
moves = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
nbrs = {(x, y): [(x + dx, y + dy) for dx, dy in moves if 0 <= x + dx < 8 and 0 <= y + dy < 8] for x in range(8) for y in range(8)}
def knight_walk(n, start=(0, 0)):
    pos = start; corner = 0; returns = []; last = 0
    for t in range(1, n + 1):
        nb = nbrs[pos]
        pos = nb[rng.integers(len(nb))]
        if pos == start:
            returns.append(t - last); last = t
    return np.array(returns)
r = knight_walk(2_000_000)
print('mean return time to the corner:', r.mean(), ' (168 expected)')
print('fraction of time in the corner:', len(r) / 2_000_000, '  2/336 =', 2 / 336)
print('median return time:', np.median(r), '  P(return within 10 moves):', np.mean(r <= 10))

The mean is $168$, but the median is only about $100$: almost a quarter of the returns happen within $10$ moves, while some excursions last thousands of moves. The mean return time is a poor summary of a skewed law, a point worth remembering when reading benchmark averages.

**P8.3.** We store the web as an edge list and multiply by $P$ with `np.add.at`, never forming the $2000\times2000$ matrix.

In [ ]:
N, k = 2000, 3
src, dst = [], []
indeg = np.zeros(N)
for v in range(1, N):
    m = min(k, v)
    p = (indeg[:v] + 1) / (indeg[:v] + 1).sum()
    targets = rng.choice(v, size=m, replace=False, p=p)
    for u in targets:
        src.append(v); dst.append(u); indeg[u] += 1
src, dst = np.array(src), np.array(dst)
outdeg = np.bincount(src, minlength=N)
alpha = 0.85
def step(x):
    y = np.zeros(N)
    np.add.at(y, dst, x[src] / outdeg[src])
    dangling = x[outdeg == 0].sum()                        # pages with no links: jump uniformly
    return alpha * (y + dangling / N) + (1 - alpha) / N
x = np.full(N, 1 / N)
hist = []
for t in range(100):
    x = step(x)
    hist.append(x)
pi = hist[-1]
errs = [0.5 * np.abs(h - pi).sum() for h in hist[:60]]
plt.semilogy(errs, label='TV distance to PageRank'); plt.semilogy([0.85**t for t in range(60)], '--', label='0.85^t')
plt.legend(); plt.xlabel('iteration'); plt.show()
top_pr = np.argsort(-pi)[:10]; top_in = np.argsort(-indeg)[:10]
print('top 10 by PageRank:', top_pr)
print('top 10 by in-degree:', top_in)
print('overlap:', len(set(top_pr) & set(top_in)), ' Spearman-like rank correlation:',
      np.corrcoef(np.argsort(np.argsort(-pi)), np.argsort(np.argsort(-indeg)))[0, 1])

The error decays at least as fast as $0.85^t$, as Exercise 8.4 promises (in practice faster). In-degree and PageRank largely agree on the top pages, which are the oldest, but PageRank also rewards being linked *from* important pages. In this model the only page without outgoing links is page $0$; the dangling-node rule redistributes its mass uniformly, which is what keeps $G$ a stochastic matrix.

**P8.4.** Sites of the two colours of a checkerboard do not interact, so all sites of one colour can be updated at once, which is exactly the same chain as updating them one by one.

In [ ]:
def ising(L, T, sweeps=1500, burn=500):
    s = rng.choice([-1, 1], size=(L, L))
    if T < 2.269:
        s[:] = 1                                     # start ordered below T_c, to avoid slow domain coarsening
    ii, jj = np.indices((L, L))
    masks = [(ii + jj) % 2 == c for c in (0, 1)]
    mags = []
    for sw in range(sweeps):
        for mask in masks:
            nb = np.roll(s, 1, 0) + np.roll(s, -1, 0) + np.roll(s, 1, 1) + np.roll(s, -1, 1)
            dE = 2 * s * nb                               # energy change of flipping each spin
            accept = (rng.random((L, L)) < np.exp(-dE / T)) & mask
            s[accept] *= -1
        if sw >= burn:
            mags.append(abs(s.mean()))
    return np.mean(mags)
Ts = np.linspace(1.5, 3.5, 17)
Tc = 2 / math.log(1 + math.sqrt(2))
m = [ising(32, T) for T in Ts]
onsager = [(1 - math.sinh(2 / T)**-4)**0.125 if T < Tc else 0 for T in Ts]
plt.plot(Ts, m, 'o-', label='Metropolis, L=32'); plt.plot(Ts, onsager, 'k--', label='Onsager (infinite lattice)')
plt.axvline(Tc, color='gray', lw=0.5); plt.xlabel('T'); plt.ylabel('|magnetisation|'); plt.legend(); plt.show()
for T, a, b in zip(Ts[::4], m[::4], onsager[::4]):
    print(f'T={T:.2f}: simulated {a:.3f}   Onsager {b:.3f}')

Below $T_c$ the simulation matches Onsager's formula closely; above it the magnetisation of a finite system does not vanish exactly (it is of order $\frac1L$), and near $T_c$ the curve is smoothed out: a phase transition is sharp only in the infinite limit. Near $T_c$ the Metropolis chain also mixes very slowly (``critical slowing down''), which is why cluster algorithms (Swendsen–Wang, Wolff) were invented.

**P8.5.** The law after $t$ steps is computed by repeated multiplication (a convolution); $t_{\mathrm{mix}}$ is the first $t$ with $d(t)\le\frac14$.

In [ ]:
def tmix_cycle(n):
    mu = np.zeros(n); mu[0] = 1
    t = 0
    while True:
        d = 0.5 * np.abs(mu - 1 / n).sum()
        if d <= 0.25:
            return t
        mu = 0.5 * mu + 0.25 * np.roll(mu, 1) + 0.25 * np.roll(mu, -1)
        t += 1
for n in [8, 16, 32, 64, 128, 256]:
    t = tmix_cycle(n)
    print(f'n={n:4d}  t_mix={t:6d}  t_mix/n^2={t/n**2:.4f}   bounds: {math.log(2)/(2*math.pi**2):.4f} <= . <= 0.25')

$\frac{t_{\mathrm{mix}}}{n^2}$ converges to about $0.095$, between the bounds $0.035$ and $0.25$ of Theorem 8.17. The limit is a property of Brownian motion on a circle: a lazy step has variance $\frac12$, so after $t=cn^2$ steps the walk, seen at scale $\frac1n$, looks like Brownian motion on a circle of length $1$ run for time $\frac c2$ (Chapter 11), and $0.095$ is the value of $c$ at which its law is at distance $\frac14$ from uniform.

**P8.6.** Under the uniform law the number of fixed points is close to Poisson$(1)$; we use the exact law, computed from the derangement numbers.

In [ ]:
n = 52
# exact law of fixed points of a uniform permutation: P(F=k) = D(n-k) / (k! (n-k)!), D = derangements
D = [1, 0]
for m in range(2, n + 1):
    D.append((m - 1) * (D[-1] + D[-2]))
pF = np.array([float(Fraction(D[n - k], math.factorial(k) * math.factorial(n - k))) for k in range(n + 1)])
def shuffle_fixed(t, reps=20000):
    decks = np.tile(np.arange(n), (reps, 1))
    rows = np.arange(reps)
    for _ in range(t):
        L = rng.integers(0, n, reps); Rr = rng.integers(0, n, reps)
        a, b = decks[rows, L].copy(), decks[rows, Rr].copy()
        decks[rows, L], decks[rows, Rr] = b, a
    return (decks == np.arange(n)).sum(axis=1)
ts = [25, 50, 75, 100, 125, 150, 200, 250, 300]
lower = []
for t in ts:
    F = shuffle_fixed(t)
    emp = np.bincount(F, minlength=n + 1)[:n + 1] / len(F)
    lower.append(0.5 * np.abs(emp - pF).sum())
    print(f't={t:4d}  lower bound on d(t) from fixed points: {lower[-1]:.3f}   mean fixed points {F.mean():.2f}')
print('1/2 n ln n =', 0.5 * n * math.log(n), '   2 n H_n =', 2 * n * sum(1 / j for j in range(1, n + 1)))

The lower bound drops from near $1$ to near $0$ between $t\approx50$ and $t\approx150$, centred close to $\frac12n\ln n\approx103$: this is the cutoff, seen through one statistic. (Below $\approx0.02$ the estimate is Monte Carlo noise.) Broder's strong stationary time, $471$ steps on average, is a safe but pessimistic upper bound, by a factor $4$ asymptotically. The truth, by Diaconis and Shahshahani, is $\frac12n\ln n$.

**P8.7.** The Eulerian numbers come from Worpitzky's identity, solved as a triangular system with exact integers.

In [ ]:
from math import comb, factorial
def eulerian(n):
    A = {}
    for a in range(1, n + 1):
        A[a] = a**n - sum(A[r] * comb(a + n - r, n) for r in range(1, a))
    return A
def riffle_tv(n, k, A):
    a = 2**k
    return float(sum(A[r] * abs(Fraction(comb(a + n - r, n), a**n) - Fraction(1, factorial(n))) for r in A) / 2)
for n in [13, 26, 52, 104, 208]:
    A = eulerian(n)
    k = 1
    while riffle_tv(n, k, A) > 0.25:
        k += 1
    print(f'n={n:4d}  shuffles needed for d <= 1/4: {k:2d}   (3/2) log2 n = {1.5 * math.log2(n):.1f}')
def gsr(deck):
    n = len(deck)
    cut = rng.binomial(n, 0.5)
    left, right = list(deck[:cut]), list(deck[cut:])
    out = []
    while left and right:                                  # drop from the bottom, proportional to sizes
        if rng.random() < len(left) / (len(left) + len(right)):
            out.append(left.pop())
        else:
            out.append(right.pop())
    out.extend(reversed(left)); out.extend(reversed(right))
    return out[::-1]
def rising(deck):
    pos = np.empty(len(deck), int); pos[np.array(deck)] = np.arange(len(deck))
    return 1 + int(np.sum(pos[1:] < pos[:-1]))
n, k, reps = 52, 4, 20000
counts = np.zeros(n + 2)
for _ in range(reps):
    d = list(range(n))
    for _ in range(k):
        d = gsr(d)
    counts[rising(d)] += 1
A = eulerian(n)
exact = {r: float(Fraction(A[r] * comb(2**k + n - r, n), 2**(k * n))) for r in range(1, 2**k + 1)}
for r in range(8, 17):
    print(f'r={r:2d}: simulated {counts[r] / reps:.4f}   exact {exact[r]:.4f}')

The number of shuffles needed follows $\frac32\log_2n$ closely, slightly below it ($8$ for $52$ cards, since $d_7=0.334$ and $d_8=0.167$, against $8.6$). After $4$ shuffles of $52$ cards there are at most $16$ rising sequences, almost always between $10$ and $16$, whereas a well-shuffled deck has about $26.5$: a simple test that any magician (or casino) can use. The simulated law matches the exact one, which checks Theorem 8.23, Proposition 8.24 and the dropping rule of Exercise 8.7 together.